# 🔨 Step 1b — 编译 llama.cpp 并存成 Kaggle Dataset

编译带 CUDA 的 `llama.cpp`，把二进制 + `.so` 存成永久私有 Dataset。只跑一次，
以后每个 session 直接挂载，省掉 26 分钟编译。

### 相对原版的两处改动

| 改动 | 原因 |
|---|---|
| 加 `-DGGML_CUDA_FA_ALL_QUANTS=ON` | `2_run_server` 默认开 `--flash-attn on` + `-ctk/-ctv q8_0`（量化 KV 缓存省显存）。量化 KV + FlashAttention 必须带上这个编译开关，否则量化后的 KV 类型用不了，甚至跑出乱码。 |
| `CMAKE_CUDA_ARCHITECTURES` 可指定 | 默认还是 `75`（T4）。以后 Kaggle 换卡（如 `89` = L4 / `80` = A100）改一行就行，不用猜。 |

> ⚠️ `.so` 必须一起打包。只拷二进制，`llama-server` 启动就会
> `cannot open shared object file`。

### 前置条件

- Accelerator → **GPU T4 x1**（CUDA 编译用）
- Internet → **ON**


## ⚙️ Cell 1 — 配置

In [ ]:
import os
from pathlib import Path

# ── 你的 Kaggle 用户名 ────────────────────────────────────────────────────────
KAGGLE_USERNAME = ""   # ← 例如 "johndoe"
# ──────────────────────────────────────────────────────────────────────────────

DATASET_SLUG = "llama-cpp-bin"

# GPU 架构：T4 = 75，L4 = 89，A100 = 80。Kaggle 免费档填 75。
CUDA_ARCH = "75"

BUILD_DIR  = Path("/tmp/llama_build")
OUTPUT_DIR = Path("/tmp/llama_output")

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("请先在 Cell 1 顶部填 KAGGLE_USERNAME。")

BUILD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 配置就绪")
print(f"   Kaggle 用户 : {KAGGLE_USERNAME}")
print(f"   Dataset     : {DATASET_SLUG}")
print(f"   CUDA 架构   : sm_{CUDA_ARCH}")


## 🔨 Cell 2 — 编译

关键编译开关：

```
-DGGML_CUDA=ON
-DGGML_CUDA_FA_ALL_QUANTS=ON   ← 新增：量化 KV 缓存 + FlashAttention 需要
-DCMAKE_CUDA_ARCHITECTURES=<CUDA_ARCH>
-DCMAKE_PREFIX_PATH=/usr/local/nvidia
```

> ⏳ 约 26 分钟。只有第一次或主动升级 llama.cpp 时才需要。


In [ ]:

import subprocess, textwrap

# 用 Python 拼 shell 脚本：Cell 1 的 CUDA_ARCH 是唯一来源，换卡只改一处
script = textwrap.dedent(f"""
set -e

echo "📦 安装构建依赖..."
apt-get update -qq
apt-get install -qq pciutils build-essential cmake curl libcurl4-openssl-dev

# Kaggle 的 libcuda.so 在 /usr/local/nvidia/lib64，cmake 却在 /usr/local/cuda/lib64 找
echo "🔧 修 CUDA 软链..."
ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/local/cuda/lib64/libcuda.so 2>/dev/null || true

echo "📥 克隆 llama.cpp..."
rm -rf /tmp/llama_build/llama.cpp
git clone --depth=1 https://github.com/ggml-org/llama.cpp /tmp/llama_build/llama.cpp

cd /tmp/llama_build/llama.cpp
COMMIT=$(git rev-parse --short HEAD)
DATE=$(git log -1 --format='%ci')
echo "   Commit : $COMMIT  ($DATE)"
echo "$COMMIT" > /tmp/llama_build/commit.txt
echo "$DATE"   >> /tmp/llama_build/commit.txt

echo "⚙️  配置 CUDA sm_{CUDA_ARCH} + FA all quants..."
cmake -B build -DGGML_CUDA=ON -DGGML_CUDA_FA_ALL_QUANTS=ON -DGGML_NATIVE=OFF -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} -DCMAKE_PREFIX_PATH=/usr/local/nvidia 2>&1 | tail -8

echo "🔨 编译（~26 分钟）..."
cmake --build build -j$(nproc) 2>&1 | tail -8

echo ""
echo "✅ 编译完成"
ls -lh build/bin/llama-server build/bin/llama-cli build/bin/llama-mtmd-cli
find build -name '*.so*' -not -path '*/CMakeFiles/*' 2>/dev/null | head -20
""")

print(script)
r = subprocess.run(["bash", "-lc", script], text=True)
assert r.returncode == 0, "❌ 编译失败，看上面的输出"

print()
print("✅ build 目录: /tmp/llama_build/llama.cpp/build")


## 📋 Cell 3 — 收集二进制 + 动态库

`llama-server` 动态链接 `libllama-common.so` 等，二进制和 `.so` 必须放一起。


In [ ]:
import shutil, subprocess, json, os
from pathlib import Path

BIN_DIR    = Path("/tmp/llama_build/llama.cpp/build/bin")
BUILD_ROOT = Path("/tmp/llama_build/llama.cpp/build")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = BIN_DIR / name
    assert src.exists(), f"缺二进制: {src}"
    dst = OUTPUT_DIR / name
    shutil.copy2(src, dst)
    dst.chmod(0o755)

so_files = [f for f in BUILD_ROOT.rglob("*.so*")
            if "CMakeFiles" not in str(f) and f.is_file()]
for so in so_files:
    shutil.copy2(so, OUTPUT_DIR / so.name)

os.environ["LD_LIBRARY_PATH"] = str(OUTPUT_DIR)

commit_lines = Path("/tmp/llama_build/commit.txt").read_text().strip().splitlines()
commit_hash  = commit_lines[0]
commit_date  = commit_lines[1] if len(commit_lines) > 1 else "unknown"

ver = subprocess.run(
    [str(OUTPUT_DIR / "llama-server"), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": str(OUTPUT_DIR)},
)
version_str = (ver.stdout or ver.stderr).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "unknown"

version_info = {
    "llama_cpp_commit": commit_hash,
    "commit_date":      commit_date,
    "llama_server_ver": version_str,
    "cuda_arch":        f"sm_{CUDA_ARCH} (T4)",
    "build_flags":      "GGML_CUDA=ON GGML_CUDA_FA_ALL_QUANTS=ON",
    "built_on":         "Kaggle",
    "ld_library_path":  "/kaggle/working",
}
(OUTPUT_DIR / "version.json").write_text(json.dumps(version_info, indent=2), encoding="utf-8")

print("📦 产物：\n")
for f in sorted(OUTPUT_DIR.iterdir()):
    mb = f.stat().st_size / 1024**2
    tag = "← binary" if f.name in binaries else ("← shared lib" if ".so" in f.name else "")
    print(f"   {f.name:<40} {mb:8.1f} MB  {tag}")

print("\n📋 version.json：")
for k, v in version_info.items():
    print(f"   {k:<20} : {v}")


## 📦 Cell 4 — 创建 / 更新 Dataset

In [ ]:
import json, subprocess

meta = {
    "title":     "llama-cpp-bin",
    "id":        f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses":  [{"name": "mit"}],
    "isPrivate": True,
}
(OUTPUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True, text=True,
)

if check.returncode == 0:
    print("🔄 已存在 → 推新版本")
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR),
         "-m", f"llama.cpp {version_info['llama_cpp_commit']} + FA_ALL_QUANTS",
         "--dir-mode", "skip"],
        capture_output=True, text=True,
    )
else:
    print(f"✨ 新建 {KAGGLE_USERNAME}/{DATASET_SLUG}")
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "--dir-mode", "skip"],
        capture_output=True, text=True,
    )

print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")
